# Feature Engineering: `fp_enriched.csv`

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='broadband-data')
print('Connected to BigQuery: broadband-data')

Connected to BigQuery: broadband-data


In [2]:
# Load all data sources
df_projects = client.query("""
SELECT project_id, state, bead_support, estimated_miles_aerial_fiber,
       estimated_miles_buried_fiber, estimated_jobs, project_type, priority_broadband_project
FROM `broadband-data.fp_approved.deployment_projects`
""").to_dataframe()

df_locations = client.query("""
SELECT *
FROM `broadband-data.fp_approved.project_loc_counts`
""").to_dataframe()

state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New_Hampshire': 'NH',
    'New_Jersey': 'NJ', 'New_Mexico': 'NM', 'New_York': 'NY', 'North_Carolina': 'NC',
    'North_Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode_Island': 'RI', 'South_Carolina': 'SC', 'South_Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West_Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District_of_Columbia': 'DC'
}

print(f'''Projects: {df_projects.shape},
Locations: {df_locations.shape}
''')

Projects: (5462, 8),
Locations: (5462, 5)



In [3]:
df_technology = client.query("""
SELECT *
FROM `broadband-data.fp_approved.loc_proj_subgrantee`
""").to_dataframe()

print(f'Total locations: {len(df_technology)}')
print(f'Location without valid provider id:')
print(df_technology.provider_id.isna().value_counts())

Total locations: 3670423
Location without valid provider id:
provider_id
False    3088239
True      582184
Name: count, dtype: int64


In [4]:
df_incumbency = client.query("""
    WITH loc_block_lookup AS (
      SELECT
        location_id,
        block_geoid,
        ROW_NUMBER() OVER (
          PARTITION BY location_id
          ORDER BY period DESC
        ) AS rn
      FROM (
        SELECT DISTINCT location_id, block_geoid, period
        FROM `broadband-data.fcc_bdc.nbm_hive`
        WHERE location_id IN (
          SELECT location_id FROM `broadband-data.fp_approved.loc_proj_subgrantee`
        )
      )
    ),

    resolved_locs AS (
      SELECT location_id, block_geoid
      FROM loc_block_lookup
      WHERE rn = 1
    ),

    block_level_total AS (
      SELECT block_geoid, COUNT(DISTINCT location_id) AS n_locations
      FROM `broadband-data.fcc_bdc.nbm_hive`
      WHERE period = '2024_12'
      GROUP BY block_geoid
    ),

    block_level_isp_count AS (
      SELECT block_geoid, provider_id, COUNT(DISTINCT location_id) AS provider_count
      FROM `broadband-data.fcc_bdc.nbm_hive`
      WHERE period = '2024_12'
      GROUP BY block_geoid, provider_id
    )

    SELECT
        s.location_id,
        s.project_id,
        s.state,
        s.provider_id,
        r.block_geoid,
        blt.n_locations AS n_locs,
        COALESCE(b.provider_count, 0) AS present_locations,
        100.0 * COALESCE(b.provider_count, 0) / blt.n_locations AS prov_coverage
    FROM `broadband-data.fp_approved.loc_proj_subgrantee` s
    JOIN resolved_locs r USING (location_id)
    JOIN block_level_total blt USING (block_geoid)
    LEFT JOIN block_level_isp_count b
        ON b.block_geoid = r.block_geoid
        AND b.provider_id = s.provider_id
""").to_dataframe()

print(f'Awardee ISP incumbency dataframe: {df_incumbency.shape}')

Awardee ISP incumbency dataframe: (3669042, 8)


In [5]:
# create competition dataset
df_competition = df_incumbency.merge(df_technology,
    on = ['location_id', 'provider_id', 'state', 'project_id'],
    how = 'right')

# aggregate to project-level
proj_comp = df_competition.groupby('project_id').agg(
    prov_coverage = ('prov_coverage', 'mean'),
    provider_id = ('provider_id', 'first'),
    frn = ('frn', 'first'),
    n_blocks = ('block_geoid', 'nunique'),
    n_locs = ('location_id', 'count')
).reset_index()

In [6]:
df_dominant_tech = client.query("""
SELECT project_id, COUNT(*) AS funded_locations,
    SAFE_CAST(APPROX_TOP_COUNT(CAST(technology AS STRING),1)[OFFSET(0)].value AS FLOAT64) AS technology
FROM `broadband-data.fp_approved.locations`
GROUP BY project_id
""").to_dataframe()

In [7]:
tech_map = {
    60: 61,   # NGSO mistakenly encoded as GSO
    70: 71,   # group all as fw
    72: 71,   # group all as fw
}
df_technology['awarded_technology'] = df_technology['technology'].replace(tech_map)

assert df_technology['awarded_technology'].isin([40, 50, 61, 71, 0]).all() 

tech_dummies = pd.get_dummies(df_technology['awarded_technology'], prefix='tech', dtype=int)
df_tech_encoded = pd.concat([tech_dummies, df_technology], axis=1)

tech_codes = [40, 50, 61, 71]

# create dataframe with fraction of tech for each project
tech_shares = (                             
    df_tech_encoded                                                                                                          
    .groupby('project_id')[[f'tech_{c}' for c in tech_codes]]                                                         
    .mean()                                                                                                             
    .add_prefix('frac_')
    .reset_index()                                                                                                      
) 

In [8]:
# competition dataset
df_comp = client.query("""
SELECT
    project_id,
    AVG(IF(competitor_present_qualifying, 1.0, 0.0))           AS frac_competitor_qualifying,
    AVG(IF(competitor_present_qualifying_any_tech, 1.0, 0.0))  AS frac_competitor_qualifying_any,
    AVG(IF(present_at_location, 1.0, 0.0))                     AS frac_present_at_location,
    AVG(IF(present_at_location_qualifying, 1.0, 0.0))          AS frac_present_at_location_qualifying,
    AVG(IF(fiber_competitor_present, 1.0, 0.0))                AS frac_fiber_comp,
    AVG(IF(cable_competitor_present, 1.0, 0.0))                AS frac_cable_comp,
    AVG(IF(wireless_competitor_present, 1.0, 0.0))             AS frac_fw_comp,
    AVG(IF(wired_competitor_present_qualifying, 1.0, 0.0))     AS frac_wired_qualifying,
FROM `broadband-data.fp_approved.competition_dec2024`
GROUP BY project_id
""").to_dataframe()

In [9]:
# load nn dataset 
df_distance = pd.read_csv("../fp_nearest_dist.csv")
df_distance['frn'] = df_distance['frn'].astype('Int64').astype(str).str.zfill(10)
df_distance['no_prior_presence'] = df_distance['nearest_own_dist'].isna().astype(int)

# merge to get project_id
df_distance_fp = df_technology[['state','location_id','frn','provider_id','project_id']].merge(df_distance,
    on = ['state', 'location_id', 'frn', 'provider_id'],
    how = 'left'
)

# aggregate to project level for rows with valid nn distance
df_distance_fp_notna = df_distance_fp[df_distance_fp['no_prior_presence'] == 0].groupby('project_id').agg(
    nearest_mi_served_same_isp = ('nearest_own_dist', 'mean'),
    nearest_mi_served_any      = ('nearest_any_dist', 'mean')
)

# find percentage of locations without any nn
df_nn = df_distance_fp.groupby('project_id').agg(
    no_prior_presence_frac = ('no_prior_presence', 'mean'),
    n_locations = ('location_id', 'count'),
)

# join the two datasets
df_nn = df_nn.join(df_distance_fp_notna, how='left').reset_index()


In [10]:
# county-level datasets
# RUCC (long --> wide)
df_rucc_raw = pd.read_csv('../Ruralurbancontinuumcodes2023.csv', encoding='ISO-8859-1')
df_rucc_wide = df_rucc_raw.pivot_table(
    index=['FIPS','State','County_Name'], columns='Attribute', values='Value', aggfunc='first'
).reset_index()
df_rucc_wide.columns.name = None
df_rucc_wide['county_id'] = df_rucc_wide['FIPS'].astype(str).str.zfill(5)
df_rucc_wide['RUCC_2023'] = pd.to_numeric(df_rucc_wide['RUCC_2023'], errors='coerce')
df_rucc = df_rucc_wide[['county_id','RUCC_2023']].rename(columns={'RUCC_2023':'rucc_code'})

# Natural amenities: topography
df_topo_raw = pd.read_excel('../natamenf.xls').iloc[103:]
df_topo_raw.columns = df_topo_raw.iloc[0]
df_topo_raw = df_topo_raw[1:].reset_index(drop=True)
df_topo_raw['county_id'] = df_topo_raw['FIPS Code'].astype(str).str.zfill(5)
df_topo_raw = df_topo_raw.rename(columns={'TOPOG - Z':'topography_z','Water area * 100':'water_area_pct'})
df_topo = df_topo_raw[['county_id','topography_z']].copy()
df_topo['topography_z'] = pd.to_numeric(df_topo['topography_z'], errors='coerce')

# Road ruggedness
df_rugged_raw = pd.read_csv('../ruggedness-scales-2020-tracts.csv', encoding='ISO-8859-1')
df_rugged_raw['county_id'] = df_rugged_raw['CountyFIPS23'].astype(str).str.zfill(5)
df_rugged = df_rugged_raw.groupby('county_id')[['RRS']].mean().reset_index()

# ACS poverty
df_pov_raw = pd.read_csv(
    '../ACSDT5Y2024.B17001_2026-07-15T133739/ACSDT5Y2024.B17001-Data.csv'
).iloc[1:]
df_pov_raw['county_id'] = df_pov_raw['GEO_ID'].str[-5:]
df_pov_raw['B17001_001E'] = pd.to_numeric(df_pov_raw['B17001_001E'], errors='coerce')
df_pov_raw['B17001_002E'] = pd.to_numeric(df_pov_raw['B17001_002E'], errors='coerce')
df_pov_raw['poverty_rate'] = df_pov_raw['B17001_002E'] / df_pov_raw['B17001_001E'].replace(0, np.nan)
df_poverty = df_pov_raw[['county_id','poverty_rate']].dropna()

# Population density (replaces state_population)
df_housing = pd.read_excel('../2020_UA_COUNTY.xlsx')
df_housing['STATE']  = df_housing['STATE'].astype(str).str.zfill(2)
df_housing['COUNTY'] = df_housing['COUNTY'].astype(str).str.zfill(3)
df_housing['county_id'] = df_housing['STATE'] + df_housing['COUNTY']
# POPDEN_RUR = rural pop density, POPDEN_COU = county-wide pop density
df_housing = df_housing[['county_id','POPDEN_RUR','POPDEN_COU']].dropna(subset=['county_id'])

# Combined county table
df_county = (
    df_rucc
    .merge(df_topo,    on='county_id', how='outer')
    .merge(df_rugged,  on='county_id', how='outer')
    .merge(df_poverty, on='county_id', how='outer')
    .merge(df_housing, on='county_id', how='outer')
)

print(f'County table: {df_county.shape}')
print(df_county[['rucc_code','topography_z','RRS','poverty_rate','POPDEN_RUR','POPDEN_COU']].isna().sum())

County table: (3249, 7)
rucc_code        16
topography_z    138
RRS             105
poverty_rate     27
POPDEN_RUR       15
POPDEN_COU       15
dtype: int64


In [11]:
# Location -> county mapping for geographic wmeans
df_loc_county = client.query("""
SELECT loc.project_id,
    SUBSTR(nbm.block_geoid, 1, 5) AS county_id,
    COUNT(DISTINCT loc.location_id) AS location_count
FROM `broadband-data.fp_approved.locations` loc
JOIN `broadband-data.fcc_bdc.nbm_hive` nbm ON loc.location_id = nbm.location_id
WHERE nbm.period = '2024_12'
GROUP BY loc.project_id, county_id
""").to_dataframe()

In [12]:
GEO_COLS = ['rucc_code','topography_z','RRS','poverty_rate','POPDEN_RUR','POPDEN_COU']
df_county[GEO_COLS] = df_county[GEO_COLS].apply(lambda s: s.fillna(s.median()))

df_lc = df_loc_county.merge(df_county[['county_id'] + GEO_COLS], on='county_id', how='left')

for col in GEO_COLS:
    df_lc[f'{col}_x_n'] = df_lc[col].fillna(df_lc[col].median()) * df_lc['location_count']

proj_geo = df_lc.groupby('project_id').agg(
    geo_n      = ('location_count', 'sum'),
    n_counties = ('county_id', 'nunique'),
    **{f'{c}_x_n': (f'{c}_x_n', 'sum') for c in GEO_COLS}
).reset_index()

for col in GEO_COLS:
    proj_geo[f'{col}_wmean'] = proj_geo[f'{col}_x_n'] / proj_geo['geo_n']
    proj_geo.drop(columns=[f'{col}_x_n'], inplace=True)

print(f'Geographic project features: {proj_geo.shape}')

Geographic project features: (5402, 9)


In [13]:
# Merge + feature engineering
print(f"After df_projects: {len(df_projects)}")
df = df_projects.merge(df_locations, on='project_id', how='left')
print(f"After df_locations merge: {len(df)}")
df = df.merge(df_comp, on='project_id', how='left')
print(f"After df_comp merge: {len(df)}")
df = df.merge(df_dominant_tech, on='project_id', how='left')
print(f"After df_dominant_tech merge: {len(df)}")
df = df.merge(proj_comp, on='project_id', how='left')
print(f"After proj_comp merge: {len(df)}")
df = df.merge(tech_shares, on='project_id', how='left')
print(f"After tech_shares merge: {len(df)}")
df = df.merge(proj_geo, on='project_id', how='left')
print(f"After proj_geo merge: {len(df)}")
df = df.merge(df_nn, on='project_id', how='left')
print(f"After df_nn merge: {len(df)}")

aerial = df['estimated_miles_aerial_fiber'].fillna(0)
buried = df['estimated_miles_buried_fiber'].fillna(0)
df['n_total'] = df['n_total'].fillna(0)
df['n_loc'] = df['n_loc_only'].fillna(0)

df['total_fiber_miles']  = aerial + buried
df['pct_aerial']         = (aerial / df['total_fiber_miles'].replace(0, np.nan)).fillna(0)
df['miles_per_location'] = (df['total_fiber_miles'] / df['n_loc'].replace(0, np.nan)).fillna(0)
df['pct_cai']            = (df['n_cai_only'] / df['n_total'].replace(0, np.nan)).fillna(0)

df['jobs_per_location'] = (df['estimated_jobs'] / df['n_loc'].replace(0, np.nan)).fillna(0)
df['has_incumbent'] = (df['prov_coverage'] > 0).astype(int)

df['funding_per_location'] = df['bead_support'] / df['n_total'].replace(0, np.nan)  # n_total, not funded_locations
df = df.dropna(subset=['funding_per_location'])

low = df['funding_per_location'].quantile(0.025)
high = df['funding_per_location'].quantile(0.975)
df['log_funding'] = np.log1p(df['funding_per_location'])
df = df.sort_values(by='project_id', ascending=True).reset_index(drop=True)
df_trimmed = df[(df['funding_per_location'] >= low) & (df['funding_per_location'] <= high)].copy()
print(f'Samples: {df.shape[0]}, Trimmed samples: {df_trimmed.shape[0]}')

After df_projects: 5462
After df_locations merge: 5462
After df_comp merge: 5462
After df_dominant_tech merge: 5462
After proj_comp merge: 5462
After tech_shares merge: 5462
After proj_geo merge: 5462
After df_nn merge: 5462
Samples: 5462, Trimmed samples: 5188


In [14]:
df_trimmed = df_trimmed.reset_index(drop=True)

In [15]:
# ── Feature set (19 features) ────────────────────────────────────────────────
# Dropped: frac_fiber_comp, frac_cable_comp, frac_fw_comp (redundant with tech fracs)
FEATURE_COLS = [
    # Technology mix
    'frac_tech_40',
    'frac_tech_50',
    'frac_tech_61',
    'frac_tech_71',
    # Project-level
    'miles_per_location',
    'pct_aerial',
    'pct_cai',
    'n_total',
    # Competition (Dec 2024 NBM)
    'frac_competitor_qualifying',
    'frac_wired_qualifying',
    # ISP incumbency / coverage
    'prov_coverage',
    'frac_present_at_location',
    'nearest_mi_served_same_isp',
    'nearest_mi_served_any',
    'no_prior_presence_frac',
    # Geography — county-level weighted means
    'POPDEN_RUR_wmean',
    'POPDEN_COU_wmean',
    'rucc_code_wmean',
    'poverty_rate_wmean',
]

pretty_names_map = {
    'frac_tech_40':               'frac_tech_40 (HFC)',
    'frac_tech_50':               'frac_tech_50 (Fiber)',
    'frac_tech_61':               'frac_tech_61 (LEO)',
    'frac_tech_71':               'frac_tech_71 (FW)',
    'miles_per_location':         'miles_per_loc',
    'pct_aerial':                 'pct_aerial',
    'pct_cai':                    'pct_cai',
    'n_total':                    'n_total',
    'frac_competitor_qualifying': 'frac_q_comp',
    'frac_wired_qualifying':      'frac_q_wired',
    'prov_coverage':              'isp_coverage',
    'frac_present_at_location':   'frac_presence',
    'nearest_mi_served_same_isp': 'nearest_own_mi',
    'nearest_mi_served_any':      'nearest_any_mi',
    'no_prior_presence_frac':     'no_prior_presence',
    'POPDEN_RUR_wmean':           'rur_popden',
    'POPDEN_COU_wmean':           'cou_popden',
    'rucc_code_wmean':            'rucc_code',
    'poverty_rate_wmean':         'poverty_rate',
}

X = df_trimmed[FEATURE_COLS].fillna(0)
y = df_trimmed['log_funding']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf = RandomForestRegressor(n_estimators=1000, min_samples_split=2, min_samples_leaf=1,
                            max_features=0.3, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

baseline_r2   = r2_score(y_test, y_pred)
baseline_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
baseline_mae  = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(rf, X, y, cv=kf, scoring='r2')

print(f'Dataset size: {X.shape[0]}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Test R²  (log): {baseline_r2:.4f}')
print(f'CV R²   (log): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'RMSE (real $): ${baseline_rmse:,.0f}')
print(f'MAE  (real $): ${baseline_mae:,.0f}')


Dataset size: 5188
Features: 19
Test R²  (log): 0.7187
CV R²   (log): 0.7104 ± 0.0154
RMSE (real $): $2,645
MAE  (real $): $1,546


In [16]:
df.to_parquet('fp_bead_enriched.parquet', index=False)

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold, GridSearchCV, RandomizedSearchCV
# Tuned RF with enhanced features
rf_param_grid = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [20],
    'max_features': [0.3, 0.5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

X = df_trimmed[FEATURE_COLS].fillna(0)
y = df_trimmed['log_funding']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf_enh_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid, cv=kf, scoring='r2',
    n_jobs=1, verbose=0
)
rf_enh_search.fit(X_train, y_train)

print('Best RF+interactions hyperparameters:')
for k, v in rf_enh_search.best_params_.items():
    print(f'  {k}: {v}')

best_model = rf_enh_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f"Test Score: {test_score:.4f}")

Best RF+interactions hyperparameters:
  max_depth: 20
  max_features: 0.5
  min_samples_leaf: 2
  min_samples_split: 2
  n_estimators: 1000
Test Score: 0.7140


In [ ]:
# Tuned RF with enhanced features
rf_param_grid = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [None, 10, 15, 20, 30],
    'max_features': [0.3, 0.5, 0.7, 1.0],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
}

X = df_trimmed[FEATURE_COLS].fillna(0)
y = df_trimmed['log_funding']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf_rand_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid, n_iter=40, cv=kf, scoring='r2',
    random_state=42, n_jobs=1, verbose=0
)
rf_rand_search.fit(X_train, y_train)

print('Best RF+interactions hyperparameters:')
for k, v in rf_rand_search.best_params_.items():
    print(f'  {k}: {v}')

best_rand_model = rf_rand_search.best_estimator_
test_score = best_rand_model.score(X_test, y_test)
print(f"Test Score: {test_score:.4f}")

Best RF+interactions hyperparameters:
  n_estimators: 500
  min_samples_split: 2
  min_samples_leaf: 2
  max_features: 0.5
  max_depth: 20
Test Score: 0.7140
